In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, recall_score, f1_score, precision_score, balanced_accuracy_score
from scipy.stats import entropy
import matplotlib.pyplot as plt
import warnings

In [ ]:
def predictive_evaluation(df):
    y_true = df['predicted trace type']
    y_pred = df['trace type']
    return accuracy_score(y_true, y_pred), precision_score(y_true, y_pred), recall_score(y_true, y_pred), f1_score(y_true, y_pred), balanced_accuracy_score(y_true, y_pred), recall_score(y_true, y_pred, pos_label=0)

In [ ]:
def double_fault(predictions, y_true):
    nr_samples = predictions.shape[0]
    nr_estimators = predictions.shape[1]
    
    errors = predictions != y_true[:, np.newaxis]

    fault_matrix = np.zeros((nr_estimators, nr_estimators))
    for i in range(nr_estimators):
        for j in range(i+1,nr_estimators):
            pairwise = np.logical_and(errors[:, i], errors[:, j])
            ratio = np.sum(pairwise)/nr_samples
            fault_matrix[i,j] = ratio

    upper_triangle = np.triu(fault_matrix, k=1)

    return upper_triangle.sum() / (nr_estimators*(nr_estimators-1)/2)

def prediction_entropy(predictions):
    nr_samples = predictions.shape[0]

    entropies = np.zeros(nr_samples)
    for i, sample_preds in enumerate(predictions):
        _, counts = np.unique(sample_preds, return_counts=True)
        p = counts/counts.sum()
        entropies[i] = entropy(p, base=2.0)
    
    return entropies.mean()

def agreement_connectivity(predictions):
    nr_samples = predictions.shape[0]

    agreement = (predictions[:,:,None] == predictions[:, None, :])
    agreement = agreement.sum(axis=0)

    A = agreement/nr_samples
    D = np.diag(A.sum(axis=1))
    L = D-A

    eigvals, _ = np.linalg.eigh(L)

    return eigvals[1]

def variety_evaluation(predictions, y_true):
    return double_fault(predictions, y_true), prediction_entropy(predictions), agreement_connectivity(predictions)

In [ ]:
nr_files = 20
nr_iters = 10

edsm_accuracy, edsm_precision, edsm_recall, edsm_f1, edsm_ba, edsm_spec = np.zeros(nr_files), np.zeros(nr_files), np.zeros(nr_files), np.zeros(nr_files), np.zeros(nr_files), np.zeros(nr_files)

walk_accuracy, walk_precision, walk_recall, walk_f1, walk_ba, walk_spec = np.zeros((nr_iters, nr_files)), np.zeros((nr_iters, nr_files)), np.zeros((nr_iters, nr_files)), np.zeros((nr_iters, nr_files)), np.zeros((nr_iters, nr_files)), np.zeros((nr_iters, nr_files))
walk_df, walk_entropy, walk_ac = np.zeros((nr_iters, nr_files)), np.zeros((nr_iters, nr_files)), np.zeros((nr_iters, nr_files))

balanced_accuracy,balanced_precision, balanced_recall, balanced_f1, balanced_ba, balanced_spec = np.zeros((nr_iters, nr_files)), np.zeros((nr_iters, nr_files)), np.zeros((nr_iters, nr_files)), np.zeros((nr_iters, nr_files)), np.zeros((nr_iters, nr_files)), np.zeros((nr_iters, nr_files))
balanced_df, balanced_entropy, balanced_ac = np.zeros((nr_iters, nr_files)), np.zeros((nr_iters, nr_files)), np.zeros((nr_iters, nr_files))

prune_accuracy, prune_precision, prune_recall, prune_f1, prune_ba, prune_spec = np.zeros((nr_iters, nr_files)), np.zeros((nr_iters, nr_files)), np.zeros((nr_iters, nr_files)), np.zeros((nr_iters, nr_files)), np.zeros((nr_iters, nr_files)), np.zeros((nr_iters, nr_files))
prune_df, prune_entropy, prune_ac = np.zeros((nr_iters, nr_files)), np.zeros((nr_iters, nr_files)), np.zeros((nr_iters, nr_files))

sizes = np.zeros(nr_files)
sparsities = np.zeros(nr_files)

for i in range(1, nr_files+1):
    edsm_aggregate = pd.read_csv(f'edsm_runs/edsm{i}.final.json.result', sep=';')
    edsm_aggregate.columns = [col.strip() for col in edsm_aggregate.columns]
    sizes[i-1] = len(edsm_aggregate)
    sparsities[i-1] = sum(edsm_aggregate['trace type'])/len(edsm_aggregate)
    edsm_accuracy[i-1], edsm_precision[i-1], edsm_recall[i-1], edsm_f1[i-1], edsm_ba[i-1], edsm_spec[i-1] = predictive_evaluation(edsm_aggregate)
    y_true = edsm_aggregate['trace type'].values

    for j in range(1, nr_iters+1):
        balanced_aggregate = pd.read_csv(f'bmte/ensemble_runs{j}/ensemble{i}.random.json.result', sep=';')
        balanced_aggregate.columns = [col.strip() for col in balanced_aggregate.columns]
        balanced_accuracy[j-1][i-1], balanced_precision[j-1][i-1], balanced_recall[j-1][i-1], balanced_f1[j-1][i-1], balanced_ba[j-1][i-1], balanced_spec[j-1][i-1] = predictive_evaluation(balanced_aggregate)

        balanced_individual = pd.read_csv(f'bmte/ensemble_runs{j}/ensemble{i}.random.json.individual', sep=';').values[:, 2:]
        balanced_df[j-1][i-1], balanced_entropy[j-1][i-1], balanced_ac[j-1][i-1] = variety_evaluation(balanced_individual, y_true)

        walk_aggregate = pd.read_csv(f'rwalks/walk_runs{j}/rw{i}.final.random.json.result', sep=';')
        walk_aggregate.columns = [col.strip() for col in walk_aggregate.columns]
        walk_accuracy[j-1][i-1], walk_precision[j-1][i-1], walk_recall[j-1][i-1], walk_f1[j-1][i-1], walk_ba[j-1][i-1], walk_spec[j-1][i-1] = predictive_evaluation(walk_aggregate)

        walk_individual = pd.read_csv(f'rwalks/walk_runs{j}/rw{i}.final.random.json.individual', sep=';').values[:, 2:]
        walk_df[j-1][i-1], walk_entropy[j-1][i-1], walk_ac[j-1][i-1] = variety_evaluation(walk_individual, y_true)

        prune_aggregate = pd.read_csv(f'pruning_runs/prune_runs{j}/prune{i}.random.json.result', sep=';')
        prune_aggregate.columns = [col.strip() for col in prune_aggregate.columns]
        prune_accuracy[j-1][i-1], prune_precision[j-1][i-1], prune_recall[j-1][i-1], prune_f1[j-1][i-1], prune_ba[j-1][i-1], prune_spec[j-1][i-1] = predictive_evaluation(prune_aggregate)

        prune_individual = pd.read_csv(f'pruning_runs/prune_runs{j}/prune{i}.random.json.individual', sep=';').values[:, 2:]
        prune_df[j-1][i-1], prune_entropy[j-1][i-1], prune_ac[j-1][i-1] = variety_evaluation(prune_individual, y_true)
        

In [ ]:
balanced_accuracy_avg = np.mean(balanced_accuracy, axis=0)
balanced_precision_avg = np.mean(balanced_precision, axis=0)
balanced_recall_avg = np.mean(balanced_recall, axis=0)
balanced_f1_avg = np.mean(balanced_f1, axis=0)
balanced_ba_avg = np.mean (balanced_ba, axis=0)
balanced_spec_avg = np.mean(balanced_spec, axis=0)

balanced_df_avg = np.mean(balanced_df, axis=0)
balanced_entropy_avg = np.mean(balanced_entropy, axis=0)
balanced_ac_avg = np.mean(balanced_ac, axis=0)

walk_accuracy_avg = np.mean(walk_accuracy, axis=0)
walk_precision_avg = np.mean(walk_precision, axis=0)
walk_recall_avg = np.mean(walk_recall, axis=0)
walk_f1_avg = np.mean(walk_f1, axis=0)
walk_ba_avg = np.mean (walk_ba, axis=0)
walk_spec_avg = np.mean(walk_spec, axis=0)

walk_df_avg = np.mean(walk_df, axis=0)
walk_entropy_avg = np.mean(walk_entropy, axis=0)
walk_ac_avg = np.mean(walk_ac, axis=0)

prune_accuracy_avg = np.mean(prune_accuracy, axis=0)
prune_precision_avg = np.mean(prune_precision, axis=0)
prune_recall_avg = np.mean(prune_recall, axis=0)
prune_f1_avg = np.mean(prune_f1, axis=0)
prune_ba_avg = np.mean (prune_ba, axis=0)
prune_spec_avg = np.mean(prune_spec, axis=0)

prune_df_avg = np.mean(prune_df, axis=0)
prune_entropy_avg = np.mean(prune_entropy, axis=0)
prune_ac_avg = np.mean(prune_ac, axis=0)

In [ ]:
balanced_recall_sd = np.std(balanced_recall, axis=0)
balanced_ba_sd = np.std(balanced_ba, axis=0)
balanced_spec_sd = np.std(balanced_spec, axis=0)
balanced_df_sd = np.std(balanced_df, axis=0)
balanced_entropy_sd = np.std(balanced_entropy, axis=0)
balanced_ac_sd = np.std(balanced_ba, axis=0)

walk_recall_sd = np.std(walk_recall, axis=0)
walk_ba_sd = np.std(walk_ba, axis=0)
walk_spec_sd = np.std(walk_spec, axis=0)
walk_df_sd = np.std(walk_df, axis=0)
walk_entropy_sd = np.std(walk_entropy, axis=0)
walk_ac_sd = np.std(walk_ac, axis=0)

prune_recall_sd = np.std(prune_recall, axis=0)
prune_ba_sd = np.std(prune_ba, axis=0)
prune_spec_sd = np.std(prune_spec, axis=0)
prune_df_sd = np.std(prune_df, axis=0)
prune_entropy_sd = np.std(prune_entropy, axis=0)
prune_ac_sd = np.std(prune_ac, axis=0)

In [ ]:
ba_stats = pd.DataFrame()
ba_stats.index = [f"Dataset #{i}" for i in range(1,21)]
ba_stats['EDSM'] = edsm_ba
ba_stats['Random Walk (Mean)'] = walk_ba_avg
ba_stats['Random Walk (S.D.)'] = walk_ba_sd
ba_stats['BMTE (Mean)'] = balanced_ba_avg
ba_stats['BMTE (S.D.)'] = balanced_ba_sd
ba_stats['BMTE+Pruning (Mean)'] = prune_ba_avg
ba_stats['BMTE+Pruning (S.D.)'] = prune_ba_sd

ba_stats.to_csv('results/balanced_acc.csv')

In [ ]:
sensitivity_stats = pd.DataFrame()
sensitivity_stats.index = [f"Dataset #{i}" for i in range(1,21)]
sensitivity_stats['EDSM'] = edsm_recall
sensitivity_stats['Random Walk (Mean)'] = walk_recall_avg
sensitivity_stats['Random Walk (S.D.)'] = walk_recall_sd
sensitivity_stats['BMTE (Mean)'] = balanced_recall_avg
sensitivity_stats['BMTE (S.D.)'] = balanced_recall_sd
sensitivity_stats['BMTE+Pruning (Mean)'] = prune_recall_avg
sensitivity_stats['BMTE+Pruning (S.D.)'] = prune_recall_sd

sensitivity_stats.to_csv('results/sensitivity.csv')

In [ ]:
spec_stats = pd.DataFrame()
spec_stats.index = [f"Dataset #{i}" for i in range(1,21)]
spec_stats['EDSM'] = edsm_spec
spec_stats['Random Walk (Mean)'] = walk_spec_avg
spec_stats['Random Walk (S.D.)'] = walk_spec_sd
spec_stats['BMTE (Mean)'] = balanced_spec_avg
spec_stats['BMTE (S.D.)'] = balanced_spec_sd
spec_stats['BMTE+Pruning (Mean)'] = prune_spec_avg
spec_stats['BMTE+Pruning (S.D.)'] = prune_spec_sd

spec_stats.to_csv('results/specificity.csv')

In [ ]:
df_stats = pd.DataFrame()
df_stats.index = [f"Dataset #{i}" for i in range(1,21)]
df_stats['Random Walk (Mean)'] = walk_df_avg
df_stats['Random Walk (S.D.)'] = walk_df_sd
df_stats['BMTE (Mean)'] = balanced_df_avg
df_stats['BMTE (S.D.)'] = balanced_df_sd
df_stats['BMTE+Pruning (Mean)'] = prune_df_avg
df_stats['BMTE+Pruning (S.D.)'] = prune_df_sd

df_stats.to_csv('results/double_fault.csv')

In [ ]:
entropy_stats = pd.DataFrame()
entropy_stats.index = [f"Dataset #{i}" for i in range(1,21)]

entropy_stats['Random Walk (Mean)'] = walk_entropy_avg
entropy_stats['Random Walk (S.D.)'] = walk_entropy_sd
entropy_stats['BMTE (Mean)'] = balanced_entropy_avg
entropy_stats['BMTE (S.D.)'] = balanced_entropy_sd
entropy_stats['BMTE+Pruning (Mean)'] = prune_entropy_avg
entropy_stats['BMTE+Pruning (S.D.)'] = prune_entropy_sd

entropy_stats.to_csv('results/entropy.csv')

In [ ]:
ac_stats = pd.DataFrame()
ac_stats.index = [f"Dataset #{i}" for i in range(1,21)]
ac_stats['Random Walk (Mean)'] = walk_ac_avg
ac_stats['Random Walk (S.D.)'] = walk_ac_sd
ac_stats['BMTE (Mean)'] = balanced_ac_avg
ac_stats['BMTE (S.D.)'] = balanced_ac_sd
ac_stats['BMTE+Pruning (Mean)'] = prune_ac_avg
ac_stats['BMTE+Pruning (S.D.)'] = prune_ac_sd

ac_stats.to_csv('results/agreement_connectivity.csv')

In [ ]:
index = np.argsort(np.array(sparsities))
sorted_sparsities = np.sort(np.array(sparsities))

In [ ]:
index = np.argsort(np.array(sparsities))
sorted_sparsities = np.sort(np.array(sparsities))

In [ ]:
plt.figure(figsize=(18, 8))

plt.plot(sorted_sparsities, edsm_accuracy[index], color='blue', marker='o', label='EDSM')
plt.plot(sorted_sparsities, walk_accuracy_avg[index], color='green', marker='o', label='Random Walk')
plt.plot(sorted_sparsities, balanced_accuracy_avg[index], color='red', marker='o', label='Balanced Merge Tree Exploration (BMTE)')
plt.plot(sorted_sparsities, prune_accuracy_avg[index], color='darkorange', marker='o', label='BMTE+Pruning')

plt.xlabel('Density')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(18, 8))

plt.plot(sorted_sparsities, edsm_precision[index], color='blue', marker='o', label='EDSM')
plt.plot(sorted_sparsities, walk_precision_avg[index], color='green', marker='o', label='Random Walk')
plt.plot(sorted_sparsities, balanced_precision_avg[index], color='red', marker='o', label='Balanced Merge Tree Exploration (BMTE)')
plt.plot(sorted_sparsities, prune_precision_avg[index], color='darkorange', marker='o', label='BMTE+Pruning')


plt.xlabel('Density')
plt.ylabel('Precision')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(18, 8))

plt.plot(sorted_sparsities, edsm_recall[index], color='blue', marker='o', label='EDSM')
plt.plot(sorted_sparsities, walk_recall_avg[index], color='green', marker='o', label='Random Walk')
plt.plot(sorted_sparsities, balanced_recall_avg[index], color='red', marker='o', label='Balanced Merge Tree Exploration (BMTE)')
plt.plot(sorted_sparsities, prune_recall_avg[index], color='darkorange', marker='o', label='BMTE+Pruning')


plt.xlabel('Density')
plt.ylabel('Recall')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(18, 8))

plt.plot(sorted_sparsities, edsm_f1[index], color='blue', marker='o', label='EDSM')
plt.plot(sorted_sparsities, walk_f1_avg[index], color='green', marker='o', label='Random Walk')
plt.plot(sorted_sparsities, balanced_f1_avg[index], color='red', marker='o', label='Balanced Merge Tree Exploration (BMTE)')
plt.plot(sorted_sparsities, prune_f1_avg[index], color='darkorange', marker='o', label='BMTE+Pruning')

plt.xlabel('Density')
plt.ylabel('F1')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(18, 8))

plt.plot(sorted_sparsities, edsm_ba[index], color='blue', marker='o', label='EDSM')
plt.plot(sorted_sparsities, walk_ba_avg[index], color='green', marker='o', label='Random Walk')
plt.plot(sorted_sparsities, balanced_ba_avg[index], color='red', marker='o', label='Balanced Merge Tree Exploration (BMTE)')
plt.plot(sorted_sparsities, prune_ba_avg[index], color='darkorange', marker='o', label='BMTE+Pruning')

plt.xlabel('Density')
plt.ylabel('Balaned Accuracy')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(18, 8))

plt.plot(sorted_sparsities, walk_df_avg[index], color='green', marker='o', label='Random Walk')
plt.plot(sorted_sparsities, balanced_df_avg[index], color='red', marker='o', label='Balanced Merge Tree Exploration (BMTE)')
plt.plot(sorted_sparsities, prune_df_avg[index], color='darkorange', marker='o', label='BMTE+Pruning')

plt.xlabel('Density')
plt.ylabel('Double Fault Ratio (Pairwise Average)')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(18, 8))

plt.plot(sorted_sparsities, walk_entropy_avg[index], color='green', marker='o', label='Random Walk')
plt.plot(sorted_sparsities, balanced_entropy_avg[index], color='red', marker='o', label='Balanced Merge Tree Exploration (BMTE)')
plt.plot(sorted_sparsities, prune_entropy_avg[index], color='darkorange', marker='o', label='BMTE+Pruning')

plt.xlabel('Density')
plt.ylabel('Entropy')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(18, 8))

plt.plot(sorted_sparsities, walk_ac_avg[index], color='green', marker='o', label='Random Walk')
plt.plot(sorted_sparsities, balanced_ac_avg[index], color='red', marker='o', label='Balanced Merge Tree Exploration (BMTE)')
plt.plot(sorted_sparsities, prune_ac_avg[index], color='darkorange', marker='o', label='BMTE+Pruning')

plt.xlabel('Density')
plt.ylabel('Agreement Connectivity')
plt.legend()
plt.show()